# veracode_lite — pre-flight scan exploration

Run a Veracode-style scan on this repo before you submit, and explore findings interactively. Drop `veracode_lite.py` in the repo root and open this notebook alongside it.

**Confidence buckets:**
- **likely_real** (≥ 0.70) — fix or carefully justify before scan
- **needs_review** (0.30–0.70) — eyeball each one
- **likely_false_positive** (< 0.30) — likely safe; standard mitigation should clear it

**Each finding carries:**
- `cwe` / `name` / `severity` (V1–V5) / `confidence` (0–1) / `confidence_label`
- `file` / `line` / `snippet` — where it is
- `description` — what the CWE is, in plain English
- `impact` — what an attacker can do with it
- `message` — the per-instance rule message
- `signals` — score breakdown (`explain()` shows this)
- `suggested_fix` — remediation pattern
- `mitigation_template` — pre-written justification text for FP-prone CWEs

In [ ]:
from veracode_lite import Scanner

scanner = Scanner('.').run()
scanner.summary()

## CWE catalog — what types of issues turned up

Quick orientation: the unique CWE types this scan found, with their definitions. Read this first to know what you're dealing with before drilling into individual findings.

In [ ]:
import textwrap

by_cwe = {}
for f in scanner.findings:
    by_cwe.setdefault(f.cwe, []).append(f)

for cwe in sorted(by_cwe, key=lambda c: (-max(f.severity for f in by_cwe[c]), -len(by_cwe[c]))):
    findings = by_cwe[cwe]
    sev = max(f.severity for f in findings)
    n_real = sum(1 for f in findings if f.confidence_label == 'likely_real')
    print(f'{cwe} ({findings[0].name})  V{sev}  — {len(findings)} findings ({n_real} likely_real)')
    if findings[0].description:
        for line in textwrap.wrap(findings[0].description, width=92,
                                  initial_indent='    ', subsequent_indent='    '):
            print(line)
    print()

## Top findings — Veracode-style card view

The top likely-real findings printed with full context: what the CWE means, what an attacker could do, the offending line, and the suggested fix. Closest equivalent to opening a finding in the Veracode UI.

In [ ]:
def print_card(f, width=92):
    bar = '─' * width
    print(bar)
    print(f'  {f.cwe}  ({f.name})')
    print(f'  V{f.severity}  conf={f.confidence:.2f}  [{f.confidence_label}]')
    print(f'  {f.file}:{f.line}')
    print(bar)
    if f.description:
        print('  WHAT:')
        for line in textwrap.wrap(f.description, width=width-4,
                                  initial_indent='    ', subsequent_indent='    '):
            print(line)
        print()
    if f.impact:
        print('  IMPACT:')
        for line in textwrap.wrap(f.impact, width=width-4,
                                  initial_indent='    ', subsequent_indent='    '):
            print(line)
        print()
    print('  CODE:')
    print(f'    > {f.snippet}')
    print()
    print(f'  RULE FINDING: {f.message}')
    if f.suggested_fix:
        print()
        print('  FIX:')
        for line in f.suggested_fix.splitlines():
            print(f'    {line}')
    print()

# Top 5 likely_real findings
top = [f for f in scanner.findings if f.confidence_label == 'likely_real'][:5]
for f in top:
    print_card(f)

## Full DataFrame view
All findings with the new `description` and `impact` columns. `description` is truncated for screen fit.

In [ ]:
import pandas as pd

df = scanner.to_dataframe()

compact = df[['cwe', 'severity', 'confidence', 'confidence_label',
              'file', 'line', 'snippet']].copy()
compact['description'] = df['description'].str.slice(0, 80) + '...'
compact

## Triage view 1 — what to fix
High severity + high confidence. Treat these as real issues.

In [ ]:
fix_now = df[(df.confidence >= 0.70) & (df.severity >= 4)]
fix_now[['cwe', 'name', 'severity', 'confidence', 'file', 'line', 'snippet']]

## Triage view 2 — what to mitigate
High severity + low confidence. The most common Veracode false-positive zone — pre-write mitigation justifications.

In [ ]:
pre_mitigate = df[(df.confidence < 0.30) & (df.severity >= 4)]
pre_mitigate[['cwe', 'name', 'file', 'line', 'snippet', 'confidence']]

## Triage view 3 — what to look at
The middle band. Manual review will tell you which way each goes.

In [ ]:
review = df[(df.confidence >= 0.30) & (df.confidence < 0.70)]
review.sort_values(['severity', 'confidence'], ascending=False)[
    ['cwe', 'severity', 'confidence', 'file', 'line', 'snippet']
].head(30)

## CWE × confidence — where the noise is concentrated

In [ ]:
pd.crosstab(df.cwe, df.confidence_label, margins=True)

## Drill into a single finding
`explain()` shows description, impact, signal-by-signal score breakdown, and suggested fix in one view.

In [ ]:
if scanner.findings:
    scanner.explain(scanner.findings[0])

In [ ]:
# Or: pass a DataFrame row
if not df.empty:
    scanner.explain(df.iloc[0])

## Per-file hotspots
Files with the most findings. Refactor candidates.

In [ ]:
df.groupby('file').agg(
    findings=('cwe', 'count'),
    likely_real=('confidence_label', lambda s: (s == 'likely_real').sum()),
    max_severity=('severity', 'max'),
).sort_values(['likely_real', 'findings'], ascending=False).head(20)

## Export findings
JSON for sharing, CSV for ticketing. Both include description and impact, so each ticket is self-explanatory.

In [ ]:
scanner.to_json('preflight_findings.json')
df.to_csv('preflight_findings.csv', index=False)
print('Wrote: preflight_findings.json, preflight_findings.csv')

## Mitigation templates for FP-prone findings
Copy-paste ready text for the Veracode Mitigation workflow.

In [ ]:
for f in scanner.findings:
    if f.mitigation_template and f.confidence < 0.50:
        print(f'[{f.cwe}] {f.file}:{f.line}  (conf={f.confidence:.2f})')
        print(f'  > {f.snippet}')
        print(f'  Mitigation:')
        for line in f.mitigation_template.splitlines():
            print(f'    {line}')
        print()